In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.7 MB/s eta 0:00:00


In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv
from rapidfuzz import process, fuzz
import re

In [ ]:
# Obtained from https://github.com/mikeoliphant/JazzStandards
with open('JazzStandards.json') as f:
  jazz_standards = json.load(f)

# Obtained from OCR index of JRB TOCs
with open('jrb_5_index.json') as f:
  jrb_5_json = json.load(f)
with open('jrb_6_index.json') as f:
  jrb_6_json = json.load(f)

df_5 = pd.read_csv('jrb_5_index.tsv', sep='\t', names=["page_num", "song_name"], quoting=csv.QUOTE_NONE)
df_5['page_num'] = pd.to_numeric(df_5['page_num'], errors='coerce')

df_6 = pd.read_csv('jrb_6_index.tsv', sep='\t', names=["page_num", "song_name"], quoting=csv.QUOTE_NONE)
df_6['page_num'] = pd.to_numeric(df_6['page_num'], errors='coerce')


In [ ]:
def normalize_title(title):
    if not isinstance(title, str):
        return ""
    working_title = title.strip()
    working_title = working_title.upper()

    if working_title.endswith(", THE"):
      working_title = "THE " + working_title[:-5]
    elif working_title.endswith(",THE"):
      working_title = "THE " + working_title[:-4]

    working_title = re.sub(r'[^\w]', '', working_title)
    working_title = re.sub(r'\s', '', working_title)
    return working_title.strip()

# test_list = [
#     "CHEROKEE (INDIAN LOVE SONG)",
#     "GIRL FROM IPANEMA,THE",
#     "GIRL FROM IPANEMA, THE"
# ]
# for title in test_list:
#   print(title)
#   print("\t -", normalize_title(title))

In [ ]:
df_5['edition'] = 5
df_6['edition'] = 6
edition_links_df = pd.concat([df_5, df_6], ignore_index=True)

# Apply normalization to the edition titles
edition_links_df['normalized_name'] = edition_links_df['song_name'].apply(normalize_title)


# 3. Main Combination Logic
final_data = []
match_threshold = 90
manual_review_threshold = 80

In [ ]:
matched_edition_indices = set()

for standard in jazz_standards:
    standard_title = standard.get("Title")
    normalized_standard_title = normalize_title(standard_title)

    combined_song = {
        "title": standard_title,
        "composer": standard.get("Composer"),
        "key": standard.get("Key", None),
        "rhythm": standard.get("Rhythm"),
        "time_signature": standard.get("TimeSignature"),
        "found": []
    }

    if not normalized_standard_title:
      final_data.append(combined_song)
      continue

    for index, row in edition_links_df.iterrows():
      score = fuzz.token_sort_ratio(normalized_standard_title, row['normalized_name'])
      if score >= match_threshold:
        # We found a highly confident match
        print(f"Matched: {score:.2f} - {standard_title} to {row['song_name']}")
        matched_edition_indices.add(index)
        combined_song["found"].append({
            "edition_found": row['edition'],
            "page_number": row['page_num'],
            "scanned_title": row['song_name'],
            "match_score": score
        })

      elif score >= manual_review_threshold:
        print(f"\tDEBUG: {score:.2f} - {standard_title} to {row['song_name']}")

    if combined_song["found"]:
      unique_found = []
      seen = set()
      for item in combined_song["found"]:
        key = (item['edition_found'], item['page_number'])
        if key not in seen:
          seen.add(key)
          unique_found.append(item)
        else:
          print("!!!! - I ACTUALLY EXECUTE - ????")
      combined_song["found"] = unique_found
      final_data.append(combined_song)

with open('combined_songs_data.json', 'w') as f:
    json.dump(final_data, f, indent=2)

print(f"Successfully processed {len(jazz_standards)} standards.")
print(f"Output saved to combined_songs_data.json.")

# ---

Matched: 100.00 - 500 Miles High to 500 MILES HIGH
Matched: 100.00 - 502 Blues to 502 BLUES
Matched: 100.00 - 502 Blues to 502 BLUES
Matched: 100.00 - A Child Is Born to A CHILD IS BORN
Matched: 100.00 - A Child Is Born to A CHILD IS BORN
Matched: 100.00 - A Fine Romance to A FINE ROMANCE
Matched: 100.00 - A Fine Romance to A FINE ROMANCE
Matched: 100.00 - A Foggy Day to A FOGGY DAY
Matched: 100.00 - A Night In Tunisia to A NIGHT IN TUNISIA
Matched: 100.00 - A Night In Tunisia to A NIGHT IN TUNISIA
Matched: 100.00 - A Sunday Kind Of Love to A SUNDAY KIND OF LOVE
Matched: 100.00 - African Flower to AFRICAN FLOWER
Matched: 100.00 - African Flower to AFRICAN FLOWER
Matched: 100.00 - Afro Blue to AFRO BLUE
Matched: 100.00 - Afro Blue to AFRO BLUE
Matched: 100.00 - Afternoon In Paris to AFTERNOON IN PARIS
Matched: 100.00 - Afternoon In Paris to AFTERNOON IN PARIS
Matched: 100.00 - Airegin to AIREGIN
Matched: 100.00 - Airegin to AIREGIN
Matched: 100.00 - Alfie to ALFIE
Matched: 100.00 - Alic

In [ ]:
times_found = [0,0]
found_times = 0
for entry in final_data:
  if entry['found']:
    found_times += 1
  for scanned in entry["found"]:
    times_found[scanned["edition_found"]-5] += 1
print(times_found)
print(len(jrb_5_json), len(jrb_6_json))
print(len(matched_edition_indices))
print(found_times)

[276, 302]
430 399
576
343


In [ ]:
final_data

[{'title': '500 Miles High',
  'composer': 'Chick Corea',
  'key': 'Emin',
  'rhythm': 'Bossa Nova',
  'time_signature': '4/4',
  'found': [{'edition_found': 6,
    'page_number': 141,
    'scanned_title': '500 MILES HIGH',
    'match_score': 100.0}]},
 {'title': '502 Blues',
  'composer': 'Jimmy Rowles',
  'key': 'Amin',
  'rhythm': 'Waltz',
  'time_signature': '3/4',
  'found': [{'edition_found': 5,
    'page_number': 153,
    'scanned_title': '502 BLUES',
    'match_score': 100.0},
   {'edition_found': 6,
    'page_number': 142,
    'scanned_title': '502 BLUES',
    'match_score': 100.0}]},
 {'title': 'A Child Is Born',
  'composer': 'Roland Hanna',
  'key': 'Bb',
  'rhythm': 'Waltz',
  'time_signature': '3/4',
  'found': [{'edition_found': 5,
    'page_number': 2,
    'scanned_title': 'A CHILD IS BORN',
    'match_score': 100.0},
   {'edition_found': 6,
    'page_number': 79,
    'scanned_title': 'A CHILD IS BORN',
    'match_score': 100.0}]},
 {'title': 'A Fine Romance',
  'compos

In [ ]:
unmatched_index_songs = {}
for index, row in edition_links_df.iterrows():
    if index not in matched_edition_indices:
        norm_name = row['normalized_name']
        if norm_name not in unmatched_index_songs:
            unmatched_index_songs[norm_name] = {
                "title": row['song_name'],
                "locations": []
            }

        unmatched_index_songs[norm_name]["locations"].append({
            "edition_found": row['edition'],
            "page_number": row['page_num'],
            "scanned_title": row['song_name'],
            "match_score": 100
        })

unmatched_count = 0
unmatched_final_data_test = []
for norm_name, data in unmatched_index_songs.items():
    unmatched_count += 1
    fallback_song = {
        "title": data["title"],
        "composer": "No Metadata",
        "key": None,
        "rhythm": "No Metadata",
        "time_signature": "No Metadata",
        "found": data["locations"]
    }

    unmatched_final_data_test.append(fallback_song)

print(f"Fallback Process Complete. Added {unmatched_count} unique songs without metadata.")


Fallback Process Complete. Added 205 unique songs without metadata.


In [ ]:
unmatched_final_data_test

[{'title': 'A CALL FOR ALL DEMONS',
  'composer': 'No Metadata',
  'key': None,
  'rhythm': 'No Metadata',
  'time_signature': 'No Metadata',
  'found': [{'edition_found': 5,
    'page_number': 1,
    'scanned_title': 'A CALL FOR ALL DEMONS',
    'match_score': 100}]},
 {'title': 'A FAMILY JOY',
  'composer': 'No Metadata',
  'key': None,
  'rhythm': 'No Metadata',
  'time_signature': 'No Metadata',
  'found': [{'edition_found': 5,
    'page_number': 4,
    'scanned_title': 'A FAMILY JOY',
    'match_score': 100}]},
 {'title': 'ALL IN LOVE IS FAIR',
  'composer': 'No Metadata',
  'key': None,
  'rhythm': 'No Metadata',
  'time_signature': 'No Metadata',
  'found': [{'edition_found': 5,
    'page_number': 14,
    'scanned_title': 'ALL IN LOVE IS FAIR',
    'match_score': 100}]},
 {'title': 'AND NOW, THE QUEEN',
  'composer': 'No Metadata',
  'key': None,
  'rhythm': 'No Metadata',
  'time_signature': 'No Metadata',
  'found': [{'edition_found': 5,
    'page_number': 22,
    'scanned_tit

In [ ]:
df_5['normalized_name'] = df_5['song_name'].apply(normalize_title)
df_6['normalized_name'] = df_6['song_name'].apply(normalize_title)
set_ed5 = set(df_5['normalized_name'])
set_ed6 = set(df_6['normalized_name'])
added_in_ed6_norm = set_ed6 - set_ed5
removed_from_ed5_norm = set_ed5 - set_ed6
common_songs_norm = set_ed5.intersection(set_ed6)

print(f"\n[+] ADDED SONGS ({len(added_in_ed6_norm)}):")
for norm_title in sorted(list(added_in_ed6_norm)):
  original_titles = df_6[df_6['normalized_name'] == norm_title]['song_name'].unique()
  for original_title in original_titles:
      print(f"+ {original_title} (Normalized: {norm_title})")

print(f"\n[-] REMOVED SONGS ({len(removed_from_ed5_norm)}):")
for norm_title in sorted(list(removed_from_ed5_norm)):
  original_titles = df_5[df_5['normalized_name'] == norm_title]['song_name'].unique()
  for original_title in original_titles:
      print(f"- {original_title} (Normalized: {norm_title})")

print(f"\n[=] COMMON SONGS: {len(common_songs_norm)}")
print("--------------------------------------------------")


[+] ADDED SONGS (128):
+ 500 MILES HIGH (Normalized: 500MILESHIGH)
+ AGUA DE BEBER (WATER TO DRINK) (Normalized: AGUADEBEBERWATERTODRINK)
+ ALFIE (Normalized: ALFIE)
+ ALL BY MYSELF (Normalized: ALLBYMYSELF)
+ ALRIGHT, OKAY, YOU WIN (Normalized: ALRIGHTOKAYYOUWIN)
+ ALWAYS (Normalized: ALWAYS)
+ A MAN AND A WOMAN (Normalized: AMANANDAWOMAN)
+ APPLE HONEY (Normalized: APPLEHONEY)
+ A STRING OF PEARLS (Normalized: ASTRINGOFPEARLS)
+ A SUNDAY KIND OF LOVE (Normalized: ASUNDAYKINDOFLOVE)
+ BEAUTY AND THE BEAST (Normalized: BEAUTYANDTHEBEAST)
+ BLACK COFFEE (Normalized: BLACKCOFFEE)
+ BROADWAY (Normalized: BROADWAY)
+ BYRD LIKE (Normalized: BYRDLIKE)
+ CALL ME (Normalized: CALLME)
+ CALL ME IRRESPONSIBLE (Normalized: CALLMEIRRESPONSIBLE)
+ CAN'T HELPLOVIN’DAT MAN (Normalized: CANTHELPLOVINDATMAN)
+ C'EST SI BON (IT'S SO GOOD) (Normalized: CESTSIBONITSSOGOOD)
+ CHEGA DE SAUDADE (NO MORE BLUES) (Normalized: CHEGADESAUDADENOMOREBLUES)
+ CHEROKEE (INDIAN LOVE SONG) (Normalized: CHEROKEEINDIANL

In [ ]:
used_in_pass_2_indices = set()
for norm_name, data in unmatched_index_songs.items():
    for location in data["locations"]:
        pass_2_indices = set(edition_links_df.index) - matched_edition_indices
        used_in_pass_2_count = len(pass_2_indices)

total_source_index_rows = len(edition_links_df)
total_rows_accounted_for = len(matched_edition_indices) + used_in_pass_2_count

print(f"\n--- FINAL VALIDATION CHECK ---")
print(f"Total rows in source index (df_5 + df_6): {total_source_index_rows}")
print(f"Rows matched to metadata (Pass 1): {len(matched_edition_indices)}")
print(f"Rows used in fallback aggregation (Pass 2): {used_in_pass_2_count}")
print(f"TOTAL rows accounted for: {total_rows_accounted_for}")


--- FINAL VALIDATION CHECK ---
Total rows in source index (df_5 + df_6): 830
Rows matched to metadata (Pass 1): 576
Rows used in fallback aggregation (Pass 2): 254
TOTAL rows accounted for: 830


In [ ]:
final_data.extend(unmatched_final_data_test)
with open('combined_songs_data.json', 'w') as f:
    json.dump(final_data, f, indent=2)
final_data

[{'title': '500 Miles High',
  'composer': 'Chick Corea',
  'key': 'Emin',
  'rhythm': 'Bossa Nova',
  'time_signature': '4/4',
  'found': [{'edition_found': 6,
    'page_number': 141,
    'scanned_title': '500 MILES HIGH',
    'match_score': 100.0}]},
 {'title': '502 Blues',
  'composer': 'Jimmy Rowles',
  'key': 'Amin',
  'rhythm': 'Waltz',
  'time_signature': '3/4',
  'found': [{'edition_found': 5,
    'page_number': 153,
    'scanned_title': '502 BLUES',
    'match_score': 100.0},
   {'edition_found': 6,
    'page_number': 142,
    'scanned_title': '502 BLUES',
    'match_score': 100.0}]},
 {'title': 'A Child Is Born',
  'composer': 'Roland Hanna',
  'key': 'Bb',
  'rhythm': 'Waltz',
  'time_signature': '3/4',
  'found': [{'edition_found': 5,
    'page_number': 2,
    'scanned_title': 'A CHILD IS BORN',
    'match_score': 100.0},
   {'edition_found': 6,
    'page_number': 79,
    'scanned_title': 'A CHILD IS BORN',
    'match_score': 100.0}]},
 {'title': 'A Fine Romance',
  'compos